# nb02 - Batch ETL: bronze -> silver

The bronze layer is where data lands *as it arrives*: nested JSON, no guarantees, a few
poisoned rows. Silver is where it becomes **typed, deduplicated, validated** Delta tables
that analysts and downstream jobs can trust.

**This notebook's raw material** (written by `scripts/generate_bronze.py`, auto-seeded below):

| file | rows | deliberate problems |
|------|------|---------------------|
| `bronze/orders_raw/` | 60k orders / 6 files | `quantity` sometimes a *string*, ~2% duplicated `order_id`s, sparse `loyalty_tier`, untyped `vendor_meta` JSON string (kept raw for nb06) |
| `bronze/customers.json` | 10k | untrimmed emails, duplicate ids, date strings |
| `bronze/products.json` | 500 | clean-ish dimension |
| `bronze/country_region.csv` | 10 | used in nb07 |

**You will learn:** explicit schemas vs inference, ANSI-safe casting (`try_cast`),
window-based deduplication, data-quality gates, partitioned Delta writes, and why
silver tables are registered by location (`register_table`) so any fresh session can
find them again.


In [1]:
from pathlib import Path
import subprocess

from pyspark.sql import functions as F, Window

from de_stack import get_spark, BRONZE, LAKE_ROOT, register_table

spark = get_spark("nb02-bronze-to-silver", shuffle_partitions=8)

if not Path(BRONZE, "orders_raw").exists():
    print("bronze layer missing -> seeding it now (~40s)...")
    subprocess.run(["python3", "/scripts/generate_bronze.py"], check=True)
else:
    print("bronze already seeded")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/13 11:34:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


bronze already seeded


## 1. Look at the raw data before trusting anything

Schema **inference** reads ~1M rows to guess types - and here it guesses `quantity`
is a string, because some rows really do arrive as strings. That inference cost and
the wrong type are exactly why production pipelines pass an **explicit schema**.


In [2]:
raw_orders = spark.read.json(f"{BRONZE}/orders_raw")
raw_orders.printSchema()

raw_orders.select("order_id", "order_ts", "quantity", "unit_price", "total_amount", "loyalty_tier").show(6, truncate=False)

print("raw rows            :", raw_orders.count())
print("distinct order_ids  :", raw_orders.select("order_id").distinct().count())


root
 |-- channel: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_note: string (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- sku: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- risk_flags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- status: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- vendor_meta: string (nullable = true)



+--------+-------------------+--------+----------+------------+------------+
|order_id|order_ts           |quantity|unit_price|total_amount|loyalty_tier|
+--------+-------------------+--------+----------+------------+------------+
|O925833 |2026-09-15T07:23:34|1       |33.84     |33.84       |NULL        |
|O949320 |2026-10-07T06:16:51|3       |93.08     |279.24      |NULL        |
|O940468 |2026-09-16T11:39:30|1       |59.13     |59.13       |NULL        |
|O914023 |2026-10-11T20:36:59|3       |24.78     |74.34       |NULL        |
|O942701 |2026-09-22T13:02:14|1       |31.62     |31.62       |NULL        |
|O949587 |2026-09-18T20:02:18|2       |42.35     |84.7        |NULL        |
+--------+-------------------+--------+----------+------------+------------+
only showing top 6 rows


raw rows            : 60000


distinct order_ids  : 58797


## 2. Clean and type the orders

Transformations, in the order a real pipeline would apply them:

1. **timestamps** - `order_ts` arrives as an ISO-8601 UTC string.
2. **ANSI-safe casts** - `try_cast` turns poisoned `quantity` values into NULL instead of
   killing the whole job (remember: ANSI mode is on, plain `CAST` would throw).
3. **recomputed money** - `quantity * unit_price` rather than trusting `total_amount`.
4. **dedup** - `row_number()` keeps the latest record per `order_id`.
5. **partition column** - `order_date`, written with the table (needed for nb03's
   small-files exercise and nb04's OPTIMIZE).


In [3]:
latest = Window.partitionBy("order_id").orderBy(F.col("order_ts").desc())

orders = (
    raw_orders
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("quantity", F.expr("try_cast(quantity AS INT)"))
    .withColumn("total_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("_rn", F.row_number().over(latest))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print("after dedup:", orders.count())
orders.select("order_id", "order_ts", "quantity", "unit_price", "total_amount", "order_date", "loyalty_tier").show(5, truncate=False)


after dedup: 58797


+--------+-------------------+--------+----------+------------+----------+------------+
|order_id|order_ts           |quantity|unit_price|total_amount|order_date|loyalty_tier|
+--------+-------------------+--------+----------+------------+----------+------------+
|O900000 |2026-10-06 22:38:46|1       |64.82     |64.82       |2026-10-06|NULL        |
|O900001 |2026-09-15 21:47:51|1       |11.29     |11.29       |2026-09-15|NULL        |
|O900002 |2026-10-01 09:18:22|2       |103.93    |207.86      |2026-10-01|NULL        |
|O900003 |2026-09-15 21:10:54|2       |144.41    |288.82      |2026-09-15|NULL        |
|O900004 |2026-09-23 15:35:01|2       |22.78     |45.56       |2026-09-23|silver      |
+--------+-------------------+--------+----------+------------+----------+------------+
only showing top 5 rows


### How many rows did each step rescue or remove?

A pipeline that silently drops rows is a pipeline you cannot debug - always measure the
effect of every filter.


In [4]:
from pyspark.sql.functions import col

before = raw_orders.count()
dirty_qty = raw_orders.filter("NOT (quantity RLIKE '^[0-9]+$')").count()
dupes = before - raw_orders.select("order_id").distinct().count()
print(f"rows arriving          : {before}")
print(f"dirty quantity values  : {dirty_qty}  (try_cast -> NULL, row kept)")
print(f"duplicate order rows   : {dupes}  (row_number keeps the newest)")


rows arriving          : 60000
dirty quantity values  : 0  (try_cast -> NULL, row kept)
duplicate order rows   : 1203  (row_number keeps the newest)


## 3. Data-quality gates

Silver tables should refuse to publish garbage. These checks are the contract: if any
rule fails, the notebook **fails loudly** instead of serving bad data downstream.


In [5]:
rules = {
    "missing order_id":       "order_id IS NULL",
    "missing customer_id":    "customer_id IS NULL",
    "quantity out of range":  "quantity IS NOT NULL AND quantity NOT BETWEEN 1 AND 10",
    "amount not positive":    "total_amount IS NOT NULL AND total_amount <= 0",
    "unparsed timestamp":     "order_ts IS NULL",
}

# every rule counts *violations* (NULL-safe: a NULL quantity from a dirty row is a
# documented outcome of try_cast, not a violation - its total_amount is NULL too)
failures = {name: orders.filter(cond).count() for name, cond in rules.items()}
for name, n in failures.items():
    print(f"{name:25s} {n} violating rows")

assert all(n == 0 for n in failures.values()), f"DQ gate failed: {failures}"
print("\nall data-quality gates passed")


missing order_id          0 violating rows
missing customer_id       0 violating rows
quantity out of range     0 violating rows
amount not positive       0 violating rows
unparsed timestamp        0 violating rows

all data-quality gates passed


## 4. Customers & products

Dimensions get the same treatment, lighter: trim/lower emails, parse dates, dedup.
`vendor_meta` and `items` stay untouched in orders - nb06 uses them for the VARIANT demo.


In [6]:
customers = (
    spark.read.json(f"{BRONZE}/customers.json")
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("signup_date", F.to_date("signup_date"))
    .withColumn("is_active", F.col("is_active").cast("boolean"))
    .dropDuplicates(["customer_id"])
)

products = (
    spark.read.json(f"{BRONZE}/products.json")
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("cost", F.col("cost").cast("double"))
)

print("customers:", customers.count(), " products:", products.count())
customers.show(3, truncate=False)


customers: 10000  products: 500


+-----------------+-------+-----------+---------------------------+---------+------------+-------------+-----------+
|city             |country|customer_id|email                      |is_active|loyalty_tier|name         |signup_date|
+-----------------+-------+-----------+---------------------------+---------+------------+-------------+-----------+
|East Phillipburgh|IT     |C100000    |hoffmankathleen@hotmail.com|false    |bronze      |Richard Smith|2023-09-07 |
|Melissashire     |DE     |C100001    |samuel82@gmail.com         |true     |bronze      |Kristin Jones|2025-07-22 |
|Charleston       |US     |C100002    |millermaria@hotmail.com    |true     |gold        |Arthur Park  |2022-10-21 |
+-----------------+-------+-----------+---------------------------+---------+------------+-------------+-----------+
only showing top 3 rows


## 5. Publish silver as Delta

`saveAsTable` writes a **Delta** table (ACID log in `_delta_log/`, not a pile of JSON
files) partitioned by `order_date`. Note the in-memory catalog caveat: table *data*
persists under `/data/warehouse`, but session *metadata* can die with the kernel -
hence `register_table()` in the next cell, used by every later notebook.


In [7]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# orders opts into the change data feed at creation: nb04 reads row-level CDC from it,
# and recording must start when the table starts (it is a no-op cost-wise for reads)
(orders.write.format("delta").mode("overwrite")
    .partitionBy("order_date")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable("silver.orders"))

(customers.write.format("delta").mode("overwrite")
    .saveAsTable("silver.customers"))

(products.write.format("delta").mode("overwrite")
    .saveAsTable("silver.products"))

print("silver tables written")


silver tables written


In [8]:
# read back like a fresh session would
assert register_table(spark, "silver", "orders")
spark.sql("DESCRIBE TABLE silver.orders").show(30, truncate=False)

spark.sql('''
    SELECT order_date, COUNT(*) AS orders, ROUND(SUM(total_amount), 2) AS revenue
    FROM silver.orders GROUP BY order_date ORDER BY order_date LIMIT 5
''').show()


26/10/13 11:35:02 WARN CreateNamespaceExec: Namespace silver was created concurrently. Ignoring.


+-----------------------+-------------------------------------------------+-------+
|col_name               |data_type                                        |comment|
+-----------------------+-------------------------------------------------+-------+
|channel                |string                                           |NULL   |
|currency               |string                                           |NULL   |
|customer_id            |string                                           |NULL   |
|customer_note          |string                                           |NULL   |
|items                  |array<struct<price:double,qty:bigint,sku:string>>|NULL   |
|loyalty_tier           |string                                           |NULL   |
|order_id               |string                                           |NULL   |
|order_ts               |timestamp                                        |NULL   |
|payment_method         |string                                           |N

26/10/13 11:35:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+------+---------+
|order_date|orders|  revenue|
+----------+------+---------+
|2026-09-13|   929| 69184.81|
|2026-09-14|  1913|142548.17|
|2026-09-15|  1983|139680.06|
|2026-09-16|  1927|140909.95|
|2026-09-17|  1931|137870.47|
+----------+------+---------+



## Wrap-up

Silver is built: typed, deduplicated, validated, partitioned Delta tables. Next:
**nb03_performance_aqe.ipynb** - these tables were written with 6 tiny files per
partition on purpose, which is the small-files problem AQE is about to solve for us.
